In [29]:
import sys
sys.path.insert(0, "..")   # project root, so `src` can be imported

import time                        # timing training and prediction

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and pipelines
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from src.data import get_numpy_data


In [30]:
def evaluate_model(model, name):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    val_pred = model.predict(X_val)
    pred_time = time.perf_counter() - start

    val_acc = accuracy_score(y_val, val_pred)
    # prediction time in ms: fast models (e.g. logistic regression) round to 0.00 in seconds
    print(f"{name}: val acc {val_acc:.3f} | train {train_time:.1f}s | "
          f"predict {pred_time * 1000:.1f} ms ({pred_time / len(X_val) * 1000:.4f} ms/image)")
    return {"model": name, "val_acc": val_acc, "train_time": train_time, "pred_time": pred_time}

results = []   # one row per chosen model, shown as a table at the end


In [31]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = get_numpy_data()

print(X_train.shape, y_train.shape)   # (55000, 784) (55000,)
print(X_val.shape,   y_val.shape)     # (5000, 784)  (5000,)
print(X_test.shape,  y_test.shape)    # (10000, 784) (10000,)


(55000, 784) (55000,)
(5000, 784) (5000,)
(10000, 784) (10000,)


In [32]:
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])


evaluate_model(logreg, "LogReg (scaled, C=1)");   # ; hides the returned dict

print("Training set score: {:.3f}".format(logreg.score(X_train, y_train)))

LogReg (scaled, C=1): val acc 0.910 | train 6.3s | predict 9.9 ms (0.0020 ms/image)
Training set score: 0.945


In [33]:
for C in [1.0, 0.1, 0.01]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=C, max_iter=1000)),
    ])
    evaluate_model(model, f"LogReg (scaled, C={C})")
    print("Training set score: {:.3f}".format(model.score(X_train, y_train)))

LogReg (scaled, C=1.0): val acc 0.910 | train 6.0s | predict 10.2 ms (0.0020 ms/image)
Training set score: 0.945
LogReg (scaled, C=0.1): val acc 0.916 | train 4.7s | predict 10.2 ms (0.0020 ms/image)
Training set score: 0.944
LogReg (scaled, C=0.01): val acc 0.917 | train 3.5s | predict 10.1 ms (0.0020 ms/image)
Training set score: 0.937


In [34]:
logreg_c_001 = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(C=0.01, max_iter=1000)),
])

results.append(evaluate_model(logreg_c_001, "LogReg (scaled, C=0.01)"))   # chosen logistic regression

print("Training set score: {:.3f}".format(logreg_c_001.score(X_train, y_train)))

LogReg (scaled, C=0.01): val acc 0.917 | train 3.9s | predict 10.3 ms (0.0021 ms/image)
Training set score: 0.937


In [35]:
logreg_raw = LogisticRegression(max_iter=1000)

evaluate_model(logreg_raw, "LogReg (raw pixels, C=1)")

print("Training set score: {:.3f}".format(logreg_raw.score(X_train, y_train)))

LogReg (raw pixels, C=1): val acc 0.914 | train 7.4s | predict 2.7 ms (0.0005 ms/image)
Training set score: 0.940


### Logistic regression: comparison

| Model | Scaler | C | Training time | Prediction time (5k images) | Train acc | Val acc | Train–val gap | Notes |
|---|---|---|---|---|---|---|---|---|
| `logreg_raw` | No | 1.0 | 7.4s | 2.7 ms | 0.940 | 0.914 | 2.6 | Raw [0, 1] pixels; slowest to converge |
| `logreg` | StandardScaler | 1.0 | 6.3s | 10 ms | 0.945 | 0.910 | 3.5 | Scaling inflates rare edge pixels → mild overfitting |
| C loop | StandardScaler | 0.1 | 4.7s | 10 ms | 0.944 | 0.916 | 2.8 | More regularization, better val |
| `logreg_c_001` | StandardScaler | **0.01** | **3.9s** | 10 ms | 0.937 | **0.917** | **2.0** | **Best:** highest val acc, smallest gap, fastest to train |

Times vary by a second or two between runs; accuracies are identical every run (the solver is deterministic).
Prediction is a single matrix multiplication, so it takes milliseconds; the scaled versions are slightly slower
because the scaler transforms the 5k images first.

**Choice:** StandardScaler + `C=0.01`, about 91.7% validation accuracy. This is the linear baseline the other models have to beat.


In [36]:
knn = KNeighborsClassifier()   # default k=5

evaluate_model(knn, "k-NN (k=5)");

k-NN (k=5): val acc 0.971 | train 0.0s | predict 677.1 ms (0.1354 ms/image)


In [37]:
for K in [3, 5, 7, 9, 11]:
    evaluate_model(KNeighborsClassifier(n_neighbors=K), f"k-NN (k={K})")

k-NN (k=3): val acc 0.976 | train 0.0s | predict 651.3 ms (0.1303 ms/image)
k-NN (k=5): val acc 0.971 | train 0.0s | predict 654.2 ms (0.1308 ms/image)
k-NN (k=7): val acc 0.969 | train 0.0s | predict 662.3 ms (0.1325 ms/image)
k-NN (k=9): val acc 0.969 | train 0.0s | predict 652.7 ms (0.1305 ms/image)
k-NN (k=11): val acc 0.967 | train 0.0s | predict 687.4 ms (0.1375 ms/image)


In [38]:
knn_3 = KNeighborsClassifier(n_neighbors=3)

evaluate_model(knn_3, "k-NN (k=3)")

k-NN (k=3): val acc 0.976 | train 0.0s | predict 656.3 ms (0.1313 ms/image)


{'model': 'k-NN (k=3)',
 'val_acc': 0.9758,
 'train_time': 0.016301399969961494,
 'pred_time': 0.6562573999981396}

In [39]:
for n in [30, 50, 100]:
    model = Pipeline([("pca", PCA(n_components=n, random_state=42)),
                      ("knn", KNeighborsClassifier(n_neighbors=3))])
    evaluate_model(model, f"PCA({n}) + k-NN (k=3)")

PCA(30) + k-NN (k=3): val acc 0.975 | train 0.4s | predict 53.7 ms (0.0107 ms/image)
PCA(50) + k-NN (k=3): val acc 0.977 | train 0.4s | predict 71.7 ms (0.0143 ms/image)
PCA(100) + k-NN (k=3): val acc 0.977 | train 0.3s | predict 99.1 ms (0.0198 ms/image)


In [40]:
knn_3_50 = Pipeline([("pca", PCA(n_components=50, random_state=42)),
                     ("knn", KNeighborsClassifier(n_neighbors=3))])

results.append(evaluate_model(knn_3_50, "knn (k=3) and (PCA=50)"))  # chosen knn

knn (k=3) and (PCA=50): val acc 0.977 | train 0.3s | predict 68.1 ms (0.0136 ms/image)


### k-NN: comparison

| Model | PCA | k | Training time | Prediction time (5k images) | Per image | Val acc | Notes |
|---|---|---|---|---|---|---|---|
| `knn` | No | 5 (default) | ~0s | 0.68s | 0.135 ms | 0.971 | Baseline |
| k loop | No | **3** | ~0s | 0.65s | 0.130 ms | **0.976** | Best k: smaller k follows the closest examples more closely |
| k loop | No | 7 / 9 / 11 | ~0s | 0.65–0.69s | ~0.13 ms | 0.969 / 0.969 / 0.967 | Larger k averages over more neighbors and blurs the boundaries between digits |
| PCA loop | 30 | 3 | 0.4s | 0.054s | 0.011 ms | 0.975 | Slightly too few components |
| `knn_3_50` | **50** | **3** | **0.3s** | **0.068s** | **0.014 ms** | **0.977** | **Best:** top accuracy, ~10× faster prediction than raw pixels |
| PCA loop | 100 | 3 | 0.3s | 0.099s | 0.020 ms | 0.977 | Same accuracy as 50, but slower |

- **Training time ≈ 0** without PCA: k-NN only stores the 55k training images. All the work happens at prediction,
  when each image is compared with every stored one. With PCA, "training" is fitting the PCA.
- **PCA helps both ways:** distances on 50 numbers instead of 784 are much cheaper, and dropping the low-variance
  components removes pixel noise, so accuracy goes slightly up.
- **No scaler:** all pixels are already on the same [0, 1] scale. Standardizing would blow up rare edge pixels
  and let them dominate the distances.
- No training accuracy: each training image is its own nearest neighbor, so it is inflated and tells us nothing.

**Choice:** PCA(50) + k-NN with k=3, about 97.7% validation accuracy. That's about 6 points above logistic regression.

In [41]:
forest = RandomForestClassifier(random_state=42)

evaluate_model(forest, "Random forest (100 trees)");

print("Training set score: {:.3f}".format(forest.score(X_train, y_train)))

Random forest (100 trees): val acc 0.967 | train 17.2s | predict 78.7 ms (0.0157 ms/image)
Training set score: 1.000


In [42]:
for n in [50, 100, 200, 300]:
    rf_trees = RandomForestClassifier(n_estimators=n, random_state=42)
    evaluate_model(rf_trees, f"Random forest ({n} trees)")

Random forest (50 trees): val acc 0.967 | train 8.5s | predict 40.6 ms (0.0081 ms/image)
Random forest (100 trees): val acc 0.967 | train 17.1s | predict 78.4 ms (0.0157 ms/image)
Random forest (200 trees): val acc 0.969 | train 33.7s | predict 152.3 ms (0.0305 ms/image)
Random forest (300 trees): val acc 0.969 | train 50.6s | predict 231.7 ms (0.0463 ms/image)


In [43]:
for n in [50, 100, 200, 300]:
    rf_trees = RandomForestClassifier(n_estimators=n, random_state=0)
    evaluate_model(rf_trees, f"Random forest ({n} trees)")

Random forest (50 trees): val acc 0.966 | train 8.6s | predict 42.2 ms (0.0084 ms/image)
Random forest (100 trees): val acc 0.968 | train 17.0s | predict 80.8 ms (0.0162 ms/image)
Random forest (200 trees): val acc 0.969 | train 33.7s | predict 152.5 ms (0.0305 ms/image)
Random forest (300 trees): val acc 0.970 | train 51.8s | predict 243.0 ms (0.0486 ms/image)


### More trees help slightly (50 trees has a validation accuracy of 0.966, while 300 has 0.970), but they drastically increase training and prediction time

In [44]:
# max_features = how many pixels each split may choose from:
# "sqrt" = 28 (default), "log2" = 9, 0.1 = 78 (10% of 784)
for n in [50, 300]:
    for max_features in ["sqrt", "log2", 0.1]:
        rf = RandomForestClassifier(n_estimators=n, max_features=max_features, random_state=42)
        evaluate_model(rf, f"Random forest ({n} trees, max_features={max_features})")

Random forest (50 trees, max_features=sqrt): val acc 0.967 | train 8.9s | predict 42.7 ms (0.0085 ms/image)
Random forest (50 trees, max_features=log2): val acc 0.962 | train 3.4s | predict 50.0 ms (0.0100 ms/image)
Random forest (50 trees, max_features=0.1): val acc 0.967 | train 24.1s | predict 38.0 ms (0.0076 ms/image)
Random forest (300 trees, max_features=sqrt): val acc 0.969 | train 53.1s | predict 244.1 ms (0.0488 ms/image)
Random forest (300 trees, max_features=log2): val acc 0.967 | train 20.4s | predict 283.9 ms (0.0568 ms/image)
Random forest (300 trees, max_features=0.1): val acc 0.970 | train 145.0s | predict 230.9 ms (0.0462 ms/image)


### The default max_features remains the best option, since training becomes too long with the other options while validation accuracy stays the same

In [45]:
forest_50 = RandomForestClassifier(n_estimators=50, random_state=42)   # default max_features="sqrt"

results.append(evaluate_model(forest_50, "Random forest (50 trees)"))   # chosen random forest

print("Training set score: {:.3f}".format(forest_50.score(X_train, y_train)))

Random forest (50 trees): val acc 0.967 | train 8.9s | predict 44.7 ms (0.0089 ms/image)
Training set score: 1.000


### Random forest: comparison

| Model | Trees | max_features | Training time | Prediction time (5k images) | Val acc (seed 42 / seed 0) | Notes |
|---|---|---|---|---|---|---|
| `forest` | 100 (default) | sqrt (28) | 17.2s | 79 ms | 0.967 / 0.968 | Baseline |
| tree loop | 50 | sqrt | 8.5s | 41 ms | 0.967 / 0.966 | Same accuracy as 100, half the cost |
| tree loop | 200 | sqrt | 33.7s | 152 ms | 0.969 / 0.969 | +0.2 points for 4× the cost of 50 |
| tree loop | 300 | sqrt | 50.6s | 232 ms | 0.969 / 0.970 | Best accuracy, 6× the cost of 50 |
| max_features loop | 50 | log2 (9) | 3.4s | 50 ms | 0.962 | Fastest to train, but too few pixels per split hurts accuracy |
| max_features loop | 50 | 0.1 (78) | 24.1s | 38 ms | 0.967 | No gain over sqrt, ~3× slower to train |
| max_features loop | 300 | log2 | 20.4s | 284 ms | 0.967 | Below sqrt at the same tree count |
| max_features loop | 300 | 0.1 | 145.0s | 231 ms | 0.970 | Ties the best accuracy at ~3× the training time of sqrt |
| `forest_50` | **50** | **sqrt** | **8.9s** | **45 ms** | **0.967** | **Chosen:** same accuracy within noise, cheapest reasonable option |

Times vary between runs (up to 2× in earlier runs); accuracies are deterministic for a given seed.

- **Seed noise is about ±0.002.** Changing `random_state` moves validation accuracy by as much as going from 50 to 300 trees,
  so the gains from more trees are mostly noise. With 5,000 validation images the standard error is also about ±0.2%.
- **Cost grows linearly with the number of trees**, both for training and prediction, and so does the saved model's size.
- **Training accuracy is 1.000 for every forest.** Each tree is grown until it fits its training sample perfectly; averaging
  many different trees is what makes the forest generalize. So the train–val gap doesn't mean overfitting here the way it does for logistic regression.
- **No scaler or PCA:** trees split on one pixel at a time using thresholds, so the scale of the features doesn't matter.

**Choice:** 50 trees with the default `max_features="sqrt"`, about 96.7% validation accuracy. That's about 5 points above logistic regression,
but 1 point below PCA + k-NN, and it takes longer to train.

In [46]:
for n in [30, 50, 100]:
    model = Pipeline([("pca", PCA(n_components=n, random_state=42)),
                      ("svc", SVC(kernel="rbf"))])
    evaluate_model(model, f"SVM RBF (PCA {n})")

SVM RBF (PCA 30): val acc 0.978 | train 7.1s | predict 2179.1 ms (0.4358 ms/image)
SVM RBF (PCA 50): val acc 0.981 | train 8.2s | predict 2240.4 ms (0.4481 ms/image)
SVM RBF (PCA 100): val acc 0.980 | train 11.5s | predict 3210.0 ms (0.6420 ms/image)


### PCA components: 50

| PCA | Training time | Prediction time (5k images) | Val acc |
|---|---|---|---|
| 30 | 7.1s | 2.2s | 0.978 |
| **50** | **8.2s** | **2.2s** | **0.981** |
| 100 | 11.5s | 3.2s | 0.980 |

- **30 is too few:** it drops some of the detail that separates similar digits, so accuracy falls by 0.3 points.
- **100 adds cost without gain:** every kernel computation is on twice as many numbers, so training and prediction are
  both about 40% slower, and accuracy is the same as 50 within noise.
- Same result as for k-NN, so both distance-based models use the same 50-component PCA.

**Choice:** PCA with 50 components. Next, C is tuned at this setting.

In [47]:
svm = Pipeline([("pca", PCA(n_components=50, random_state=42)),
                ("svc", SVC(kernel="rbf"))])

evaluate_model(svm, "SVM RBF (PCA = 50, C = 1)")

print("Training set score: {:.3f}".format(svm.score(X_train, y_train)))

SVM RBF (PCA = 50, C = 1): val acc 0.981 | train 8.0s | predict 2226.4 ms (0.4453 ms/image)
Training set score: 0.991


In [48]:
for C in [0.1, 0.5, 1, 5, 10]:
    model = Pipeline([("pca", PCA(n_components=50, random_state=42)),
                      ("svc", SVC(kernel="rbf", C=C))])
    evaluate_model(model, f"SVM RBF (PCA 50, C={C})")
    print("Training set score: {:.3f}".format(model.score(X_train, y_train)))

SVM RBF (PCA 50, C=0.1): val acc 0.962 | train 15.8s | predict 4126.0 ms (0.8252 ms/image)
Training set score: 0.969
SVM RBF (PCA 50, C=0.5): val acc 0.977 | train 9.2s | predict 2596.1 ms (0.5192 ms/image)
Training set score: 0.986
SVM RBF (PCA 50, C=1): val acc 0.981 | train 8.3s | predict 2254.3 ms (0.4509 ms/image)
Training set score: 0.991
SVM RBF (PCA 50, C=5): val acc 0.985 | train 7.2s | predict 2026.2 ms (0.4052 ms/image)
Training set score: 0.999
SVM RBF (PCA 50, C=10): val acc 0.985 | train 7.3s | predict 2030.7 ms (0.4061 ms/image)
Training set score: 1.000


### C: 5

C is the penalty for misclassified training points. Higher C = less regularization, the model follows the training data more closely.

- **Low C underfits:** C=0.1 gives 0.962 val and only 0.969 train, so the model is too constrained. Accuracy rises steadily up to C=5.
- **C=5 and C=10 tie at 0.985.** C=10 reaches 1.000 training accuracy and gains nothing on validation, so going higher only fits the training data harder.
- **Higher C was also faster** (training 15.8s → 7.2s, prediction 4.1s → 2.0s from C=0.1 to C=5). With low C, many more training points end up as
  support vectors, and prediction compares every image against each of them.

**Choice:** C=5, the smallest C that reaches the best validation accuracy, so it is the most regularized of the two tied options.
Next, gamma is tuned with C=5.

In [49]:
pca_50 = PCA(n_components=50, random_state=42).fit(X_train)
gamma_scale = 1 / (50 * pca_50.transform(X_train).var())   # what gamma="scale" uses

for factor in [0.25, 0.5, 1, 1.5, 2]:
    model = Pipeline([("pca", PCA(n_components=50, random_state=42)),
                      ("svc", SVC(kernel="rbf", C=5, gamma=gamma_scale * factor))])
    evaluate_model(model, f"SVM RBF (PCA 50, C=5, gamma={factor}x scale)")
    print("Training set score: {:.3f}".format(model.score(X_train, y_train)))


SVM RBF (PCA 50, C=5, gamma=0.25x scale): val acc 0.976 | train 6.6s | predict 1976.4 ms (0.3953 ms/image)
Training set score: 0.986
SVM RBF (PCA 50, C=5, gamma=0.5x scale): val acc 0.982 | train 6.2s | predict 1773.1 ms (0.3546 ms/image)
Training set score: 0.994
SVM RBF (PCA 50, C=5, gamma=1x scale): val acc 0.985 | train 7.4s | predict 2038.0 ms (0.4076 ms/image)
Training set score: 0.999
SVM RBF (PCA 50, C=5, gamma=1.5x scale): val acc 0.986 | train 10.2s | predict 2712.1 ms (0.5424 ms/image)
Training set score: 0.999
SVM RBF (PCA 50, C=5, gamma=2x scale): val acc 0.986 | train 14.0s | predict 3141.8 ms (0.6284 ms/image)
Training set score: 1.000


### gamma: default (`"scale"`)

gamma controls how far each support vector's influence reaches. Low gamma = wide reach and a smooth boundary;
high gamma = narrow reach and a boundary that bends around individual training points.
The values above are multiples of the default, `gamma="scale"` = 1 / (n_features × variance of the data).

- **Lower gamma underfits:** 0.25× gives 0.976 and 0.5× gives 0.982. The train–val gap shrinks, but validation accuracy drops, so the boundary is too smooth.
- **Higher gamma doesn't really help:** 1.5× and 2× reach 0.986, only 0.1 point (5 images) above the default's 0.985. That's within the
  validation noise of about ±0.2%, while training takes 40–90% longer and prediction 35–55% longer. 2× also reaches 1.000 training accuracy.
- The default heuristic already lands at the top of the accuracy curve here, so tuning gamma brings no real gain.

**Choice:** the default gamma. Final SVM: PCA(50) + RBF SVM, C=5, `gamma="scale"`.

In [50]:
svm_c5 = Pipeline([("pca", PCA(n_components=50, random_state=42)),
                   ("svc", SVC(kernel="rbf", C=5))])   # default gamma="scale"

results.append(evaluate_model(svm_c5, "SVM RBF (PCA 50, C=5)"))   # chosen SVM

print("Training set score: {:.3f}".format(svm_c5.score(X_train, y_train)))
print("Support vectors:", svm_c5.named_steps["svc"].n_support_.sum())

SVM RBF (PCA 50, C=5): val acc 0.985 | train 7.7s | predict 2070.8 ms (0.4142 ms/image)
Training set score: 0.999
Support vectors: 8896


### SVM (RBF kernel): comparison

| Model | PCA | C | gamma | Training time | Prediction time (5k images) | Train acc | Val acc | Notes |
|---|---|---|---|---|---|---|---|---|
| PCA loop | 30 | 1 | scale | 7.1s | 2.2s | – | 0.978 | Too few components |
| `svm` | 50 | 1 (default) | scale | 8.0s | 2.2s | 0.991 | 0.981 | Baseline |
| PCA loop | 100 | 1 | scale | 11.5s | 3.2s | – | 0.980 | No gain over 50, slower |
| C loop | 50 | 0.1 | scale | 15.8s | 4.1s | 0.969 | 0.962 | Too much regularization: underfits, and slowest |
| C loop | 50 | 0.5 | scale | 9.2s | 2.6s | 0.986 | 0.977 | |
| C loop | 50 | 10 | scale | 7.3s | 2.0s | 1.000 | 0.985 | Ties C=5, fits training data perfectly |
| gamma loop | 50 | 5 | 0.25× / 0.5× scale | 6.6s / 6.2s | 2.0s / 1.8s | 0.986 / 0.994 | 0.976 / 0.982 | Boundary too smooth: underfits |
| gamma loop | 50 | 5 | 1.5× / 2× scale | 10.2s / 14.0s | 2.7s / 3.1s | 0.999 / 1.000 | 0.986 / 0.986 | +0.1 point (noise) for 40–90% more training time |
| `svm_c5` | **50** | **5** | **scale** | **7.7s** | **2.1s** | **0.999** | **0.985** | **Chosen** |

- **PCA 50 is also best for the SVM**, the same as for k-NN: 30 loses a little accuracy, and 100 adds cost without gain.
- **Higher C was faster**, both to train and to predict. With low C the SVM allows many margin violations, so many more
  training points become support vectors, and prediction compares each image with every support vector.
  The chosen model keeps 8,896 of the 55,000 training images (16%) as support vectors.
- **gamma:** lower values underfit. 1.5× and 2× reach 0.986, within noise (about ±0.2%) of the default's 0.985, at a clearly higher cost.
- **No scaler:** the pixels are already in [0, 1], and `gamma="scale"` adapts to the spread of the PCA features.
- **Slow prediction:** about 0.4 ms per image, roughly 30× slower than PCA + k-NN. Still far below a millisecond for one drawn digit in the demo.

**Choice:** PCA(50) + RBF SVM with `C=5` and the default gamma, about 98.5% validation accuracy, the best classical model.
That's 0.8 points above PCA + k-NN and about 7 points above logistic regression.

In [51]:
# Comparison of the chosen models (validation set)
pd.DataFrame(results).sort_values("val_acc", ascending=False).round(3)

,model,val_acc,train_time,pred_time
3,"SVM RBF (PCA 50, C=5)",0.985,7.723,2.071
1,knn (k=3) and (PCA=50),0.977,0.282,0.068
2,Random forest (50 trees),0.967,8.853,0.045
0,"LogReg (scaled, C=0.01)",0.917,3.872,0.010


## Summary (validation set)

| Model | Settings | Val acc | Training time | Prediction time (5k images) | Per image |
|---|---|---|---|---|---|
| **SVM (RBF)** | PCA 50, C=5, gamma="scale" | **0.985** | 7.7s | 2.07s | 0.41 ms |
| k-NN | PCA 50, k=3 | 0.977 | 0.3s | 0.068s | 0.014 ms |
| Random forest | 50 trees, max_features="sqrt" | 0.967 | 8.9s | 0.045s | 0.009 ms |
| Logistic regression | StandardScaler, C=0.01 | 0.917 | 3.9s | 0.010s | 0.002 ms |

- **Non-linear models win clearly.** All three are 5–7 points above logistic regression. A linear model draws straight
  boundaries in pixel space, which can't capture the many ways each digit is written.
- **The SVM is the most accurate** (98.5%), but the slowest to predict: each image is compared with ~8,900 support vectors.
- **k-NN is the best trade-off:** only 0.8 points behind the SVM, ~30× faster to predict, and practically no training.
- **The random forest needs no preprocessing** and predicts fastest of the non-linear models, but is the least accurate of the three.
- **PCA(50) helped both distance-based models**, making them faster and slightly more accurate by removing pixel noise.
- **The model type matters far more than tuning.** Tuning improved each model by less than 1 point over its default settings,
  and many differences were within the validation noise of about ±0.2%. In those cases the cheaper option was chosen.

All numbers are on the validation set, which was used for every choice above. The test set has not been used yet.
The final comparison, including the CNN, is done once on the test set in `src/evaluate.py`.